# Final Technical Appendix

## Recreating A Stroke Pathway DES In Python And SimPy

This notebook consolidates the final technical appendix into a single end-to-end narrative. It includes environment setup guidance, reproducible scenario settings, the full sequence of simulation analyses, and the figures and tables used to compare the recreated model with Monks et al.


## Environment Setup

Recommended steps from the repository root:

```bash
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m ipykernel install --user --name hdpm097
```

The authoritative setup for the submission branch is documented in the root `README.md`.

If you open the `Codex` folder in VS Code, keep the interpreter and notebook kernel pointed at the root `.venv` before running this notebook.


## Reproducibility

This notebook uses a fixed random seed strategy so that replications are reproducible. The scenario runner starts from a known seed and increments deterministically across replications.

For the technical appendix, the replication count is set to `30` so that the notebook remains practical to run end to end. The paper used `150` replications, so that parameter can be increased later if needed.


In [ ]:
from pathlib import Path
import platform
import sys

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib
import pandas as pd
import simpy
from IPython.display import Image, display

from stroke_sim.config import SimulationSettings
from stroke_sim.metrics import (
    plot_delay_tradeoff_curve,
    plot_occupancy_distribution,
)
from stroke_sim.pooling import build_pooling_results_table
from stroke_sim.runner import build_delay_tradeoff, run_replications, run_single_replication
from stroke_sim.validation import (
    PUBLISHED_CURRENT_ADMISSIONS_ACUTE,
    PUBLISHED_CURRENT_ADMISSIONS_ACUTE_10_TO_15,
    PUBLISHED_CURRENT_ADMISSIONS_REHAB,
    PUBLISHED_MORE_ADMISSIONS_ACUTE,
    PUBLISHED_MORE_ADMISSIONS_REHAB,
    PUBLISHED_NO_COMPLEX_ACUTE,
    PUBLISHED_NO_COMPLEX_REHAB,
    PUBLISHED_RING_FENCED_ACUTE,
    build_table_2_style,
    build_two_scenario_table,
    compare_to_published_table,
)

RANDOM_SEED = 42
REPLICATIONS = 30
RUN_LENGTH_DAYS = 365 * 5
WARM_UP_DAYS = 365 * 3

print({
    'python': platform.python_version(),
    'simpy': simpy.__version__,
    'pandas': pd.__version__,
    'matplotlib': matplotlib.__version__,
    'random_seed': RANDOM_SEED,
    'replications': REPLICATIONS,
})


## Shared Scenario Settings

All scenarios below use the same run length and warm-up as the paper, with a reproducible seed schedule. We run the core scenarios once and reuse the audited outputs throughout the appendix.


In [ ]:
base_kwargs = dict(
    run_length_days=RUN_LENGTH_DAYS,
    warm_up_days=WARM_UP_DAYS,
    replications=REPLICATIONS,
)

current_settings = SimulationSettings(**base_kwargs)
more_settings = SimulationSettings(arrival_rate_multiplier=1.05, **base_kwargs)
no_complex_settings = SimulationSettings(
    excluded_patient_groups=('complex_neurological',),
    **base_kwargs,
)

current_audit = run_replications(settings=current_settings, start_seed=RANDOM_SEED)
more_audit = run_replications(settings=more_settings, start_seed=RANDOM_SEED)
no_complex_audit = run_replications(settings=no_complex_settings, start_seed=RANDOM_SEED)

single_run_outputs = run_single_replication(random_seed=RANDOM_SEED, settings=current_settings)
acute_distribution = single_run_outputs['acute_distribution']

print({
    'current_rows': len(current_audit),
    'more_rows': len(more_audit),
    'no_complex_rows': len(no_complex_audit),
})


## Base Scenario Figure 1 Style Output

This reproduces the role of the paper's occupancy distribution figure by plotting the audited daily occupancy distribution for the acute stroke unit.


In [ ]:
acute_occupancy_figure = root / 'docs' / 'figures' / 'final_appendix_acute_occupancy_distribution.png'
fig1 = plot_occupancy_distribution(
    acute_distribution,
    title='Simulation probability density function for occupancy of an acute stroke unit',
    x_label='No. patients in acute unit',
    output_path=acute_occupancy_figure,
    max_x=30,
)
fig1


In [ ]:
display(Image(filename=str(acute_occupancy_figure)))


## Base Scenario Figure 3 Style Output

This reproduces the trade-off between acute bed numbers and probability of delay using the recreated occupancy audit.


In [ ]:
acute_tradeoff = build_delay_tradeoff(current_audit, column='acute_occupancy', bed_range=range(0, 29))
acute_tradeoff_figure = root / 'docs' / 'figures' / 'final_appendix_acute_delay_tradeoff.png'
fig3 = plot_delay_tradeoff_curve(
    acute_tradeoff,
    title='Simulated trade-off between the probability that a patient is delayed and the no. of acute beds available',
    x_label='No. of acute beds available',
    output_path=acute_tradeoff_figure,
)
fig3


In [ ]:
display(Image(filename=str(acute_tradeoff_figure)))


## Validation Summary For Current Admissions

These tables compare the recreated model to the published current-admissions results.


In [ ]:
current_acute_comparison = compare_to_published_table(
    current_audit,
    column='acute_occupancy',
    published=PUBLISHED_CURRENT_ADMISSIONS_ACUTE,
)
current_rehab_comparison = compare_to_published_table(
    current_audit,
    column='rehab_occupancy',
    published=PUBLISHED_CURRENT_ADMISSIONS_REHAB,
)
current_acute_comparison


In [ ]:
current_rehab_comparison


## Scenario 1: Current Admissions Versus 5% More Admissions

This table is shaped like Table 2 in Monks et al.


In [ ]:
acute_more_comparison = compare_to_published_table(
    more_audit,
    column='acute_occupancy',
    published=PUBLISHED_MORE_ADMISSIONS_ACUTE,
)
rehab_more_comparison = compare_to_published_table(
    more_audit,
    column='rehab_occupancy',
    published=PUBLISHED_MORE_ADMISSIONS_REHAB,
)

acute_table_2 = build_table_2_style(
    current_acute_comparison,
    acute_more_comparison,
    bed_label='No. acute beds',
    include_only_more_beds={10, 11, 12, 13, 14},
)
rehab_table_2 = build_table_2_style(
    current_rehab_comparison,
    rehab_more_comparison,
    bed_label='No. rehab beds',
    include_only_more_beds={12, 13, 14, 15, 16},
)
table_2_style = pd.concat([acute_table_2, rehab_table_2], ignore_index=True)
table_2_style


## Scenario 2: Pooling Acute And Rehab Beds

This table is shaped like the paper's pooling table and uses the appendix's partial-pooling probability logic.


In [ ]:
pooling_table = build_pooling_results_table(current_audit)
pooling_table


## Scenario 3: Effect Of Complex Neurological Patients On Flow

This table is shaped like the supplementary results table comparing current admissions with no complex neurological patients.


In [ ]:
acute_no_complex_comparison = compare_to_published_table(
    no_complex_audit,
    column='acute_occupancy',
    published=PUBLISHED_NO_COMPLEX_ACUTE,
)
rehab_no_complex_comparison = compare_to_published_table(
    no_complex_audit,
    column='rehab_occupancy',
    published=PUBLISHED_NO_COMPLEX_REHAB,
)

acute_no_complex_table = build_two_scenario_table(
    current_acute_comparison,
    acute_no_complex_comparison,
    bed_label='No. acute beds',
    baseline_prefix='current',
    scenario_prefix='no_complex',
)
rehab_no_complex_table = build_two_scenario_table(
    current_rehab_comparison,
    rehab_no_complex_comparison,
    bed_label='No. rehab beds',
    baseline_prefix='current',
    scenario_prefix='no_complex',
)
supplementary_no_complex_table = pd.concat([acute_no_complex_table, rehab_no_complex_table], ignore_index=True)
supplementary_no_complex_table


## Scenario 4: Ring Fenced Acute Stroke Beds

This table compares current admissions with a ring-fenced interpretation in which stroke beds are reserved for stroke patients only. The `1 in every n` values are computed from the unrounded probabilities, even when the displayed `p(delay)` rounds to `0.00`.


In [ ]:
current_acute_10_15 = compare_to_published_table(
    current_audit,
    column='acute_occupancy',
    published=PUBLISHED_CURRENT_ADMISSIONS_ACUTE_10_TO_15,
)
ring_fenced_comparison = compare_to_published_table(
    current_audit,
    column='acute_stroke_occupancy',
    published=PUBLISHED_RING_FENCED_ACUTE,
)
ring_fenced_table = build_two_scenario_table(
    current_acute_10_15,
    ring_fenced_comparison,
    bed_label='No. acute beds',
    baseline_prefix='current',
    scenario_prefix='ring_fenced',
)
ring_fenced_table


## Output Files Generated By This Notebook

The notebook writes two figure files to `docs/figures/`:

- `final_appendix_acute_occupancy_distribution.png`
- `final_appendix_acute_delay_tradeoff.png`


In [ ]:
print({
    'acute_occupancy_figure': str(acute_occupancy_figure),
    'acute_tradeoff_figure': str(acute_tradeoff_figure),
})


## Final Notes

- This notebook is the consolidated end-to-end appendix narrative and supersedes the earlier iteration notebooks for final submission use.
- The earlier iteration notebooks remain useful as an audit trail of development and prompt-driven refinement.
- The recreated model is intentionally simpler than the original SIMUL8 implementation, so remaining mismatches should be discussed openly in the report as part of the recreation study.
